# Gaussian Process Regression for an Irregularly Sampled Stellar Light Curve


In this exercise you reconstruct an unknown stellar light curve from irregularly sampled, heteroscedastic observations using a Gaussian Process (GP). The notebook follows one continuous chain:

$$ k(t,t') \;\longrightarrow\; K_{ij}=k(t_i,t_j) \;\longrightarrow\; \text{prior functions} \;\longrightarrow\; p(\mathbf f_\star \mid \mathbf y) \;\longrightarrow\; \text{choose the next observation} \;\longrightarrow\; \text{choose the kernel.} $$


### Learning objectives
By the end you should be able to
1. interpret the amplitude $A$ and length scale $\ell$ of a squared-exponential kernel: $A^2 {\rm exp}(-\frac{t^2}{2l^2})$
2. construct and inspect a GP covariance matrix;
3. draw functions from a GP prior;
4. implement the GP predictive mean and covariance with Cholesky linear algebra;
5. distinguish uncertainty in the latent signal from uncertainty in a future noisy observation;
6. use the predictive variance to choose a useful future observing time;
7. *(extension)* compare kernels on held-out data and recognise kernel misspecification.

### Notation
$$ y_i = f(t_i) + \epsilon_i, \qquad \epsilon_i \sim N(0,\sigma_i^2), \qquad f(t) \sim GP\big(0,\, k(t,t')\big). $$

### How to use this notebook
- Cells marked `# TODO` contain blanks (`...`) for you to fill in. Plotting boilerplate is provided so you can concentrate on the GP.
- Several cells contain `assert` self-checks — if they pass, your implementation is correct.
- Questions marked **Q** should be answered in the markdown cell below them **before** moving on.
- The hidden true signal is revealed only in Part 6. Do not read the generator cell.


## 0 — Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky, cho_solve, solve_triangular

plt.rcParams['figure.figsize'] = (10, 4.5)
plt.rcParams['font.size'] = 12

# np.trapezoid is numpy >= 2.0; fall back to np.trapz on older versions
trapezoid = getattr(np, 'trapezoid', None) or np.trapz

## Instructor-provided synthetic observations

The data set imitates a variable star observed at irregular times in **two observing seasons**, separated by a seasonal gap, with point-dependent measurement uncertainties and a smooth but non-trivial latent signal.

The generator is in the cell below. **Run it, but do not read `_hidden_stellar_signal`** — it is the answer you are trying to recover and is used only in the final validation section.

The data products are `t_obs` (days), `flux` (mean-subtracted relative flux) and `yerr` (1σ uncertainty per point).


In [ ]:
# ---- hidden generator: run but do not read -----------------------------------
rng_data = np.random.default_rng(2026)

t_obs = np.sort(np.concatenate([rng_data.uniform(0.3, 7.8, 32),
                                rng_data.uniform(12.2, 19.7, 28)]))

def _hidden_stellar_signal(t):
    """Synthetic latent stellar variability — used only for validation in Part 6."""
    t = np.asarray(t, dtype=float)
    return (0.65 * np.sin(2 * np.pi * t / 6.4 + 0.25)
            + 0.22 * np.sin(2 * np.pi * t / 2.5 - 0.7)
            + 0.12 * np.cos(2 * np.pi * t / 13.0)
            + 0.012 * (t - 10.0))

yerr = 0.07 + 0.08 * rng_data.random(t_obs.size)
yerr[rng_data.choice(t_obs.size, size=7, replace=False)] += 0.10        # a few poor-quality points

raw_flux = _hidden_stellar_signal(t_obs) + rng_data.normal(0.0, yerr)
flux_offset = np.average(raw_flux, weights=1.0 / yerr**2)             # centre so a zero GP mean is sensible
flux = raw_flux - flux_offset
# -------------------------------------------------------------------------------

print(f"N = {t_obs.size} observations, {t_obs.min():.2f}–{t_obs.max():.2f} d, median sigma = {np.median(yerr):.3f}")

---
# Part 1 — Inspect the astronomical time series

Before fitting anything, look at the sampling pattern and the uncertainties.

### Exercise 1.1
1. Plot the observations with error bars (points, not lines — connecting the dots already implies a model).
2. Compute the spacings `dt = np.diff(t_obs)` and find the start and end of the **longest gap**.
3. Shade the gap on the plot.


In [ ]:
# TODO
dt = ...
i_gap = ...                      # index of the largest spacing
gap_start, gap_end = ..., ...
print(f"median dt = {np.median(dt):.2f} d,  longest gap: {gap_start:.2f}–{gap_end:.2f} d ({gap_end-gap_start:.1f} d)")

plt.errorbar(t_obs, flux, yerr=yerr, fmt='o', ms=4, capsize=2, color='k', label='observations')
plt.axvspan(gap_start, gap_end, color='C1', alpha=0.2, label='seasonal gap')
plt.xlabel('time [days]'); plt.ylabel('mean-subtracted relative flux'); plt.legend(); plt.grid(ls=':')

**Q1.1** Where is the largest gap, and how does it compare with the typical spacing?

*Your answer:*


**Q1.2** Are all measurements equally precise? How should a model weight a point with $\sigma = 0.2$ compared with one with $\sigma = 0.08$?

*Your answer:*


**Q1.3** Before computing anything: where do you expect the reconstruction uncertainty to be largest, and where smallest?

*Your answer:*


**Q1.4** Should a model of the latent stellar signal pass exactly through every point? Explain.

*Your answer:*


---
# Part 2 — From a kernel to a GP prior

We use the squared-exponential kernel
$$ k(t,t') = A^2 \exp\!\left[-\frac{(t-t')^2}{2\ell^2}\right], $$
where $A$ is the typical vertical scale of the function ($A^2$ is its variance) and $\ell$ is the characteristic correlation time scale.

## Exercise 2.1 — Implement the squared-exponential kernel

In [ ]:
def squared_exponential_kernel(t1, t2, amplitude, length_scale):
    """Return the (len(t1), len(t2)) covariance matrix K[i, j] = k(t1[i], t2[j])."""
    if amplitude <= 0 or length_scale <= 0:
        raise ValueError("amplitude and length_scale must be positive")
    t1 = np.atleast_1d(np.asarray(t1, dtype=float))
    t2 = np.atleast_1d(np.asarray(t2, dtype=float))
    # TODO: build the pairwise separation matrix dt (shape (len(t1), len(t2))) and return A^2 exp[-0.5 (dt/ell)^2]
    dt = ...
    return ...


# --- self-checks ---------------------------------------------------------------
K_test = squared_exponential_kernel(np.array([0.0, 1.0, 2.0]), np.array([0.0, 2.0]), amplitude=2.0, length_scale=1.0)
assert K_test.shape == (3, 2),                              "wrong shape"
assert np.isclose(K_test[0, 0], 4.0),                       "k(t,t) must equal A^2"
assert np.isclose(K_test[1, 0], 4.0 * np.exp(-0.5)),        "k at one length-scale must be A^2 e^{-1/2}"
assert np.isclose(K_test[0, 1], K_test[2, 0]),              "k must depend only on |t - t'|"
print("all checks passed")

## Exercise 2.2 — Plot and interpret the covariance function

Because the kernel depends only on the separation $\tau = t - t'$, plot $k(\tau)$ for $\tau \in [-5, 5]$ d using the fixed values $A = 0.7$, $\ell = 1.1$ d that we will use throughout. Mark $\pm\ell$ with vertical lines and the points $k(0)$ and $k(\ell)$.


In [ ]:
amplitude, length_scale = 0.7, 1.1          # fixed hyperparameters for the whole notebook
tau = np.linspace(-5.0, 5.0, 500)

# TODO: evaluate k(tau, 0)  — hint: squared_exponential_kernel(tau, [0.0], ...)[:, 0]
k_tau = ...
k0, k_ell = ..., ...

plt.plot(tau, k_tau)
plt.axvline(-length_scale, ls='--', color='0.5'); plt.axvline(length_scale, ls='--', color='0.5', label=r'$\pm\ell$')
plt.plot(0, k0, 'ko'); plt.annotate(rf'$k(0)=A^2={k0:.3f}$', (0, k0), xytext=(0.4, k0))
plt.plot(length_scale, k_ell, 'rs'); plt.annotate(rf'$k(\ell)={k_ell:.3f}$', (length_scale, k_ell), xytext=(length_scale + 0.4, k_ell))
plt.xlabel(r'separation $\tau = t - t^{\prime}$ [days]'); plt.ylabel(r'$k(\tau)$'); plt.legend(); plt.grid(ls=':')
print(f"k(ell) / k(0) = {k_ell / k0:.3f}")

**Q2.2** (a) What is $k(t,t)$? (b) What fraction of $k(t,t)$ remains at $|t-t'| = \ell$, and at $3\ell$? (c) Which parameter controls vertical variability and which controls how quickly the function can change?

*Your answer:*


## Exercise 2.3 — Construct and inspect a covariance matrix

Evaluate $K_{ij} = k(t_i, t_j)$ on a regular grid of 150 times on $[0, 20]$ d and display it as an image. Then check numerically that $K = K^{\mathsf T}$ and that its eigenvalues are non-negative up to floating-point precision.


In [ ]:
t_cov = np.linspace(0.0, 20.0, 150)
# TODO
K_cov = ...

plt.figure(figsize=(6, 5))
im = plt.imshow(K_cov, origin='lower', extent=[0, 20, 0, 20], cmap='viridis')
plt.colorbar(im, label=r'$K_{ij}$'); plt.xlabel(r'$t_j$ [days]'); plt.ylabel(r'$t_i$ [days]'); plt.title(r'$K_{ij}=k(t_i,t_j)$')

# TODO: symmetry error and eigenvalues (np.linalg.eigvalsh)
symmetry_error = ...
eigenvalues = ...
print(f"max |K - K^T|      = {symmetry_error:.2e}")
print(f"smallest eigenvalue = {eigenvalues.min():.2e}   largest = {eigenvalues.max():.2e}")
print(f"eigenvalues > 1e-10 : {np.sum(eigenvalues > 1e-10)} of {eigenvalues.size}")

**Q2.3** (a) Why is the diagonal the brightest region? (b) Why is there a band rather than only a thin diagonal line? (c) What would happen to the band if $\ell$ were larger? (d) Why is a tiny negative eigenvalue of order $10^{-15}$ not a physical violation, and what does it mean in practice?

*Your answer:*


## Exercise 2.4 — Draw sample functions from the GP prior

A GP prior says that the vector of function values on a grid is a draw from $N(\mathbf 0, K)$. To generate one, factorise $K + \epsilon I = L L^{T}$ (Cholesky) and set
$$ \mathbf f = L \mathbf z, \qquad \mathbf z \sim N(\mathbf 0, I). $$
(Check: $\rm{Cov}[L\mathbf z] = L L^{\mathsf T} = K$.)


In [ ]:
def draw_gp_prior(t, amplitude, length_scale, n_samples=4, rng=None, jitter=1e-10):
    """Return an array of shape (len(t), n_samples) of functions drawn from the zero-mean GP prior."""
    t = np.atleast_1d(np.asarray(t, dtype=float))
    if rng is None:
        rng = np.random.default_rng()
    # TODO: 1. construct K;  2. L = cholesky(K + jitter I, lower=True);  3. z ~ N(0, I) of shape (len(t), n_samples);  4. return L @ z
    K = ...
    L = ...
    z = ...
    return ...


t_prior = np.linspace(0.0, 20.0, 300)
prior_samples = draw_gp_prior(t_prior, amplitude, length_scale, n_samples=4, rng=np.random.default_rng(7))
assert prior_samples.shape == (300, 4)

plt.plot(t_prior, prior_samples, lw=1.3)
plt.fill_between(t_prior, -2 * amplitude, 2 * amplitude, color='0.85', alpha=0.5, label=r'prior $\pm 2A$')
plt.xlabel('time [days]'); plt.ylabel('latent flux'); plt.title('Four independent samples from the same GP prior'); plt.legend(); plt.grid(ls=':')

**Q2.4** (a) Why do the curves have different detailed shapes? (b) What statistical properties do they share? (c) Why do they look like coherent curves rather than independent random points?

*Your answer:*


## Exercise 2.5 — Change the length scale

Draw three prior samples each for $\ell = 0.35$ d and $\ell = 2.5$ d, keeping $A = 0.7$ and using the **same random numbers** (same `rng` seed) in both panels so that only the kernel changes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, ell_trial in zip(axes, [0.35, 2.5]):
    # TODO: draw 3 samples with rng=np.random.default_rng(8) and plot them
    samples = ...
    ax.plot(t_prior, samples, lw=1.3)
    ax.fill_between(t_prior, -2 * amplitude, 2 * amplitude, color='0.85', alpha=0.5)
    ax.set_title(rf'$A={amplitude},\ \ell={ell_trial}$ d'); ax.set_xlabel('time [days]'); ax.grid(ls=':')
axes[0].set_ylabel('latent flux')

**Q2.5** Explain why reducing $\ell$ makes the functions vary more rapidly even though $A$ is unchanged. Which value of $\ell$ looks more like the light curve from Part 1?

*Your answer:*


---
# Part 3 — Condition the GP on the observations

For a zero mean function, the predictive distribution of the latent function at test times $\mathbf t_\star$ is
$$ p(\mathbf f_\star \mid \mathbf y) = N(\boldsymbol\mu_\star, \mathbf C_\star), \qquad
\boldsymbol\mu_\star = \mathbf K_\star^{\mathsf T} \mathbf K_y^{-1} \mathbf y, \qquad
\mathbf C_\star = \mathbf K_{\star\star} - \mathbf K_\star^{\mathsf T} \mathbf K_y^{-1} \mathbf K_\star, $$
with $[\mathbf K_y]_{ij} = k(t_i,t_j) + \sigma_i^2 \delta_{ij}$, $[\mathbf K_\star]_{ij} = k(t_i, t_{\star j})$ and $[\mathbf K_{\star\star}]_{ij} = k(t_{\star i}, t_{\star j})$.

We never form $\mathbf K_y^{-1}$ explicitly. With the Cholesky factor $\mathbf K_y = L L^{\mathsf T}$:
$$ \boldsymbol\alpha = \mathbf K_y^{-1}\mathbf y \;(\texttt{cho\_solve}), \qquad
\boldsymbol\mu_\star = \mathbf K_\star^{\mathsf T}\boldsymbol\alpha, \qquad
\mathbf v = L^{-1}\mathbf K_\star \;(\texttt{solve\_triangular}), \qquad
\mathbf C_\star = \mathbf K_{\star\star} - \mathbf v^{\mathsf T}\mathbf v . $$

## Exercise 3.1 — Implement GP prediction


In [ ]:
def gp_predict(t_train, y_train, yerr_train, t_test, amplitude, length_scale, jitter=1e-10):
    """Return the posterior mean (n_test,) and covariance (n_test, n_test) of the latent function."""
    t_train = np.atleast_1d(np.asarray(t_train, dtype=float))
    y_train = np.atleast_1d(np.asarray(y_train, dtype=float))
    yerr_train = np.atleast_1d(np.asarray(yerr_train, dtype=float))
    t_test = np.atleast_1d(np.asarray(t_test, dtype=float))
    if not (t_train.size == y_train.size == yerr_train.size):
        raise ValueError("t_train, y_train and yerr_train must have equal length")

    # TODO
    K_y = ...                         # k(t_train, t_train) + diag(yerr^2 + jitter)
    K_star = ...                      # k(t_train, t_test)
    K_starstar = ...                  # k(t_test, t_test)
    L = ...                           # cholesky(K_y, lower=True)
    alpha = ...                       # cho_solve((L, True), y_train)
    posterior_mean = ...
    v = ...                           # solve_triangular(L, K_star, lower=True)
    posterior_covariance = ...
    posterior_covariance = 0.5 * (posterior_covariance + posterior_covariance.T)   # remove round-off asymmetry
    return posterior_mean, posterior_covariance

# --- self-checks ---------------------------------------------------------------
m, Cv = gp_predict(t_obs[:8], flux[:8], yerr[:8], np.linspace(0.0, 4.0, 25), amplitude, length_scale)
assert m.shape == (25,) and Cv.shape == (25, 25),      "wrong output shapes"
assert np.all(np.diag(Cv) <= amplitude**2 + 1e-8),     "posterior variance can never exceed the prior variance A^2"
# a single exact (sigma -> 0) observation must be reproduced at that time
m1, C1 = gp_predict([3.0], [0.4], [1e-6], [3.0, 30.0], amplitude, length_scale)
assert np.isclose(m1[0], 0.4, atol=1e-4) and np.isclose(C1[0, 0], 0.0, atol=1e-4), "exact data must be interpolated exactly"
assert np.isclose(m1[1], 0.0) and np.isclose(C1[1, 1], amplitude**2),               "far from data, revert to the prior"
print("all checks passed")

## Exercise 3.2 — Reconstruct the light curve

Evaluate the posterior on a fine grid `t_prediction` over $[0, 20]$ d with the fixed values $A = 0.7$, $\ell = 1.1$ d (supplied rather than optimised — hyperparameter training is the subject of the optional Part 7). Extract the posterior standard deviation from the diagonal of the covariance and plot the observations, the posterior mean and the latent $\pm1\sigma$ and $\pm2\sigma$ intervals.


In [ ]:
t_prediction = np.linspace(0.0, 20.0, 500)

# TODO
posterior_mean, posterior_covariance = ...
latent_std = ...                         # sqrt of the clipped (>= 0) diagonal
print(f"posterior std ranges from {latent_std.min():.3f} to {latent_std.max():.3f}  (prior: {amplitude:.3f})")


def plot_posterior(mean, std, label='posterior mean', ax=None, t=None, color='C0'):
    """Helper used repeatedly below."""
    ax = ax or plt.gca(); t = t_prediction if t is None else t
    ax.fill_between(t, mean - 2 * std, mean + 2 * std, color=color, alpha=0.15, label=r'latent $\pm2\sigma$')
    ax.fill_between(t, mean - std, mean + std, color=color, alpha=0.3, label=r'latent $\pm1\sigma$')
    ax.plot(t, mean, '-', color=color, lw=1.5, label=label)
    ax.errorbar(t_obs, flux, yerr=yerr, fmt='ko', ms=3, capsize=2, label='observations')
    ax.axvspan(gap_start, gap_end, color='C1', alpha=0.12)
    ax.set_xlabel('time [days]'); ax.set_ylabel('mean-subtracted relative flux'); ax.grid(ls=':')
    return ax

plot_posterior(posterior_mean, latent_std).legend(loc='upper right', ncol=2)
plt.title('GP reconstruction of the latent stellar light curve')

**Q3.2** (a) Where is the posterior uncertainty largest — does it match your prediction in Q1.3? (b) Why does the posterior mean not pass through every observation? (c) What happens to the prediction far from informative observations? (d) Why can a GP be confident between two closely spaced precise measurements but less confident near a single noisy one?

*Your answer:*


## Exercise 3.3 — Draw functions from the posterior

Exactly as in Part 2, but now with the posterior mean and covariance: $\mathbf f = \boldsymbol\mu_\star + L_\star \mathbf z$ with $L_\star L_\star^{\mathsf T} = \mathbf C_\star + \epsilon I$. Draw four samples and overplot the observations.

In [ ]:
rng_posterior = np.random.default_rng(11)
# TODO (use jitter 1e-9)
L_post = ...
posterior_samples = ...            # shape (500, 4)

plt.plot(t_prediction, posterior_samples, lw=1.2)
plt.errorbar(t_obs, flux, yerr=yerr, fmt='ko', ms=3, capsize=2, label='observations')
plt.axvspan(gap_start, gap_end, color='C1', alpha=0.12)
plt.xlabel('time [days]'); plt.ylabel('mean-subtracted relative flux'); plt.title('Functions sampled from the GP posterior'); plt.legend(); plt.grid(ls=':')

**Q3.3** Why do the posterior curves agree closely in some regions but differ substantially in the seasonal gap? What extra information do the samples give that the $\pm 2\sigma$ band does not?

*Your answer:*


---
# Part 4 — Latent-function uncertainty versus future-observation uncertainty

The covariance computed above describes uncertainty in the **latent function** $f_\star$. A future *measured* value is $y_\star = f_\star + \epsilon_\star$, so
$$ \mathrm{Var}(y_\star \mid \mathbf y) = \mathrm{Var}(f_\star \mid \mathbf y) + \sigma_\star^2 . $$

Assume a future measurement has the median uncertainty of the existing data. Compute the standard deviation for a future observation and plot its $\pm 2\sigma$ interval together with the latent one.


In [ ]:
sigma_future = float(np.median(yerr))
# TODO
future_observation_std = ...

ax = plot_posterior(posterior_mean, latent_std)
ax.fill_between(t_prediction, posterior_mean - 2 * future_observation_std, posterior_mean + 2 * future_observation_std,
                color='C3', alpha=0.12, label=r'future observation $\pm2\sigma$')
ax.legend(loc='upper right', ncol=2); plt.title('Two different predictive uncertainties')
print(f"assumed sigma of a future measurement: {sigma_future:.3f}")

**Q4** (a) Why is the future-observation interval wider, and where is the difference most visible? (b) Which interval should be reported for the underlying stellar variability? (c) Which should be used to forecast a future detector measurement, e.g. to decide whether a new point is anomalous?

*Your answer:*


---
# Part 5 — Choose the next observation

For fixed kernel hyperparameters the posterior covariance $\mathbf C_\star$ depends on the observation **times and uncertainties** but not on the measured **values** (look at the formula: $\mathbf y$ appears in $\boldsymbol\mu_\star$ only). We can therefore use the predictive variance to schedule a new observation before taking it. We adopt the simplest acquisition rule,
$$ t_{\rm next} = \operatorname*{arg\,max}_t \mathrm{Var}[f(t) \mid \mathbf y], $$
excluding candidate times within 0.25 d of an existing observation.

## Exercise 5.1 — Select the time


In [ ]:
candidate_times = np.linspace(0.5, 19.5, 400)
nearest = np.min(np.abs(candidate_times[:, None] - t_obs[None, :]), axis=1)
candidate_times = candidate_times[nearest > 0.25]

# TODO: posterior variance on candidate_times (values only needed on the diagonal) and argmax
candidate_variance = ...
t_next = ...
print(f"selected next observing time: {t_next:.3f} d")

plt.plot(candidate_times, candidate_variance, '.', ms=3, label=r'$\mathrm{Var}[f(t)\,|\,\mathbf{y}]$')
plt.axvline(t_next, ls='--', color='C3', label=f'selected: {t_next:.2f} d')
plt.axhline(amplitude**2, ls=':', color='0.5', label=r'prior variance $A^2$')
plt.xlabel('candidate observing time [days]'); plt.ylabel('posterior variance'); plt.legend(); plt.grid(ls=':')

**Q5.1** (a) Where was the next observation selected and why? (b) The variance curve has a flat top: what does that tell you about the choice? (c) Would changing only the measured flux values change the selected time? Verify your answer with a one-line experiment (e.g. call `gp_predict` with `2 * flux` or with `flux[::-1]`).

*Your answer:*


## Exercise 5.2 — Add the new measurement and update the GP

The cell below simulates one noisy measurement at `t_next` (the hidden signal is used here only to generate a realistic datum). Append it to the data, recompute the posterior, and compare the posterior standard deviation and the **integrated posterior variance** $\int \mathrm{Var}[f(t)\mid\mathbf y]\,dt$ before and after.


In [ ]:
sigma_next = float(np.median(yerr))
flux_next = float(_hidden_stellar_signal(t_next) + np.random.default_rng(17).normal(0.0, sigma_next) - flux_offset)
print(f"new simulated observation: t = {t_next:.3f}, y = {flux_next:.3f}, sigma = {sigma_next:.3f}")

In [ ]:
# TODO: append, sort by time, recompute posterior on t_prediction
t_aug, flux_aug, yerr_aug = ..., ..., ...
order = ...
t_aug, flux_aug, yerr_aug = t_aug[order], flux_aug[order], yerr_aug[order]

mean_aug, cov_aug = ...
std_aug = ...

int_var_before = trapezoid(latent_std**2, t_prediction)
int_var_after  = ...
print(f"integrated posterior variance: before {int_var_before:.3f}, after {int_var_after:.3f}, fraction remaining {int_var_after/int_var_before:.3f}")

plt.plot(t_prediction, latent_std, label='before new observation')
plt.plot(t_prediction, std_aug, label='after new observation')
plt.axvline(t_next, ls='--', color='C3', label='new observation')
plt.xlabel('time [days]'); plt.ylabel('posterior standard deviation'); plt.title('Effect of one targeted observation'); plt.legend(); plt.grid(ls=':')

**Q5.2** (a) Where did the uncertainty decrease most strongly, and over what range of times? How is that range related to $\ell$? (b) By how much did the integrated variance drop? (c) Why might a real observing programme use a criterion more sophisticated than maximum variance alone?

*Your answer:*


---
# Part 6 — Reveal the latent signal and assess the reconstruction

Only now do we compare the GP reconstruction with the synthetic truth. Plot the truth (minus `flux_offset`), the posterior mean before and after the new observation, the updated $\pm 2\sigma$ band and all observations; then compute the RMSE of the posterior mean against the truth over the full grid and inside the gap, before and after.


In [ ]:
true_signal = _hidden_stellar_signal(t_prediction) - flux_offset

ax = plot_posterior(mean_aug, std_aug, label='posterior mean after')
ax.plot(t_prediction, posterior_mean, 'C2-', lw=1.3, label='posterior mean before')
ax.plot(t_prediction, true_signal, 'C3--', lw=1.5, label='synthetic truth')
ax.errorbar([t_next], [flux_next], yerr=[sigma_next], fmt='o', color='C3', ms=7, capsize=3, label='new observation')
ax.legend(loc='upper right', ncol=2); plt.title('GP reconstruction compared with the synthetic truth')

gap_mask = (t_prediction > gap_start) & (t_prediction < gap_end)
rmse = lambda a, b: np.sqrt(np.mean((a - b)**2))
# TODO: four RMSE values
print(f"overall RMSE: before {...:.3f}   after {...:.3f}")
print(f"gap RMSE:     before {...:.3f}   after {...:.3f}")

# calibration: fraction of the truth inside the ±2σ band
print(f"truth inside latent 2σ band: before {np.mean(np.abs(true_signal - posterior_mean) < 2*latent_std):.2f}   after {np.mean(np.abs(true_signal - mean_aug) < 2*std_aug):.2f}   (expect ~0.95)")

**Q6.1** Did the new observation improve the RMSE in this realisation — overall and in the gap? Is the $\pm 2\sigma$ band well calibrated?

*Your answer:*


**Q6.2** Why is a reduction in posterior *variance* a more reliable observing-design objective than a guaranteed reduction in RMSE?

*Your answer:*


**Q6.3** State in one sentence each: what the kernel contributes to this analysis, and what conditioning on the observations changes.

*Your answer:*


---
# Part 7 (optional) — What if the hyperparameters are wrong?

The values $A = 0.7$, $\ell = 1.1$ d were handed to you. Everything above depends on them. Refit the posterior with $\ell = 0.35$ d and $\ell = 2.5$ d (keeping $A = 0.7$) and compare the posterior mean and band with the truth. Then compute the **log marginal likelihood**
$$ \log p(\mathbf y \mid \ell) = -\tfrac12 \mathbf y^{\mathsf T}\mathbf K_y^{-1}\mathbf y - \sum_i \log L_{ii} - \tfrac{N}{2}\log 2\pi $$
(with $L$ the Cholesky factor of $\mathbf K_y$) for a grid of $\ell$ values and see which $\ell$ the data prefer.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, ell_trial in zip(axes, [0.35, 1.1, 2.5]):
    # TODO: posterior for this ell; plot with plot_posterior(..., ax=ax) and overlay the truth
    ...
    ax.plot(t_prediction, true_signal, 'C3--', lw=1.2, label='truth')
    ax.set_title(rf'$\ell = {ell_trial}$ d'); ax.set_ylim(-1.8, 1.8)
axes[0].legend(loc='lower left', fontsize=9)


def log_marginal_likelihood(t_train, y_train, yerr_train, amplitude, length_scale, jitter=1e-10):
    # TODO
    K_y = ...
    L = ...
    alpha = ...
    return ...

ells = np.logspace(np.log10(0.2), np.log10(5.0), 60)
lml = np.array([log_marginal_likelihood(t_obs, flux, yerr, amplitude, ell) for ell in ells])
plt.figure(); plt.semilogx(ells, lml); plt.axvline(1.1, ls='--', color='0.5', label='fixed value 1.1 d')
plt.axvline(ells[np.argmax(lml)], ls=':', color='C3', label=f'maximum: {ells[np.argmax(lml)]:.2f} d')
plt.xlabel(r'$\ell$ [days]'); plt.ylabel('log marginal likelihood'); plt.legend(); plt.grid(ls=':', which='both')

**Q7** Describe the failure modes at $\ell = 0.35$ d and $\ell = 2.5$ d. Does the marginal likelihood prefer a value close to the one you were given? Why does a *too small* $\ell$ produce small residuals but a poor reconstruction?

*Your answer:*


---
# Part 8 (optional) — Kernel misspecification: which kernel predicts the missing season?

Everything so far used the squared-exponential kernel, which knows nothing about periodicity. The light curve looks roughly periodic, and a kernel that encodes that physics might predict the seasonal gap far better. But a more structured kernel can also be *over-confident* if the assumption is only approximately true. Kernel choice is a major modelling decision: use physical knowledge when you have it, combine simple components that represent the important amplitudes and time scales, and **validate on held-out data** — a flexible GP can overfit.

| Kernel | $k(\tau)$ | Prior belief |
|---|---|---|
| Squared-exponential (SE) | $A^2 \exp[-\tau^2/2\ell^2]$ | smooth, aperiodic |
| Matérn 3/2 | $A^2\left(1+\frac{\sqrt3\,|\tau|}{\ell}\right)\exp\left(-\frac{\sqrt3\,|\tau|}{\ell}\right)$ | rougher (once differentiable), aperiodic |
| Periodic | $A^2 \exp\left[-\frac{2\sin^2(\pi\tau/P)}{\Gamma^2}\right]$ | strictly periodic with period $P$ |
| Quasi-periodic | $A^2 \exp\left[-\frac{2\sin^2(\pi\tau/P)}{\Gamma^2}\right]\exp\left[-\frac{\tau^2}{2\ell_{\rm ev}^2}\right]$ | periodic, but decorrelating over $\ell_{\rm ev}$ |

We will fit each kernel (hyperparameters by maximum marginal likelihood), score it on a **withheld interval** (held-out RMSE, log predictive density, coverage), and finally compare the predictions inside the natural gap with the hidden truth.

## Exercise 8.1 — Withhold a test interval

Withhold all points with $16 \le t \le 18$ d; the rest is the training set.


In [ ]:
test_lo, test_hi = 16.0, 18.0
# TODO
test_mask = ...
t_tr, y_tr, e_tr = t_obs[~test_mask], flux[~test_mask], yerr[~test_mask]
t_te, y_te, e_te = t_obs[test_mask],  flux[test_mask],  yerr[test_mask]
print(f"train: {t_tr.size}   test: {t_te.size}")

plt.errorbar(t_tr, y_tr, yerr=e_tr, fmt='ko', ms=4, capsize=2, label='train')
plt.errorbar(t_te, y_te, yerr=e_te, fmt='o', color='C3', ms=4, capsize=2, label='withheld test')
plt.axvspan(gap_start, gap_end, color='C1', alpha=0.2, label='natural gap'); plt.axvspan(test_lo, test_hi, color='C3', alpha=0.12)
plt.xlabel('time [days]'); plt.ylabel('mean-subtracted relative flux'); plt.legend(); plt.grid(ls=':')

## Exercise 8.2 — Implement the kernels and a general predictor

Implement the three new kernels by hand, then generalise `gp_predict` and `log_marginal_likelihood` so that they accept any kernel *callable* `k(t1, t2)`.


In [ ]:
def matern32_kernel(t1, t2, amplitude, length_scale):
    t1 = np.atleast_1d(np.asarray(t1, float)); t2 = np.atleast_1d(np.asarray(t2, float))
    r = np.abs(t1[:, None] - t2[None, :]) * np.sqrt(3.0) / length_scale
    # TODO
    return ...

def periodic_kernel(t1, t2, amplitude, gamma, period):
    t1 = np.atleast_1d(np.asarray(t1, float)); t2 = np.atleast_1d(np.asarray(t2, float))
    dt = t1[:, None] - t2[None, :]
    # TODO: A^2 exp[-2 sin^2(pi dt / P) / gamma^2]
    return ...

def quasi_periodic_kernel(t1, t2, amplitude, gamma, period, ell_ev):
    # TODO: periodic (with amplitude) times a unit-amplitude SE with length ell_ev
    return ...


def gp_predict_k(t_train, y_train, yerr_train, t_test, kernel, jitter=1e-10):
    """As gp_predict, but `kernel(t1, t2)` is any covariance callable."""
    t_train = np.asarray(t_train, float); t_test = np.asarray(t_test, float)
    # TODO: same eight lines as gp_predict with squared_exponential_kernel(...) replaced by kernel(...)
    ...
    return posterior_mean, posterior_covariance


def log_marginal_likelihood_k(t_train, y_train, yerr_train, kernel, jitter=1e-10):
    # TODO
    ...


# --- self-checks ---------------------------------------------------------------
k_se = lambda a, b: squared_exponential_kernel(a, b, amplitude, length_scale)
m_ref, C_ref = gp_predict(t_obs, flux, yerr, t_prediction[:50], amplitude, length_scale)
m_new, C_new = gp_predict_k(t_obs, flux, yerr, t_prediction[:50], k_se)
assert np.allclose(m_ref, m_new) and np.allclose(C_ref, C_new), "gp_predict_k must reproduce gp_predict for the SE kernel"
assert np.isclose(log_marginal_likelihood_k(t_obs, flux, yerr, k_se), log_marginal_likelihood(t_obs, flux, yerr, amplitude, length_scale))
Kp = periodic_kernel([0.0, 6.0, 3.0], [0.0], 1.0, 1.0, 6.0)
assert np.isclose(Kp[1, 0], Kp[0, 0]), "periodic kernel must be exactly periodic: k(P) = k(0)"
assert Kp[2, 0] < Kp[0, 0],             "half a period away the correlation must be lower"
Km = matern32_kernel([0.0, 1.0], [0.0], 2.0, 1.0)
assert np.isclose(Km[0, 0], 4.0) and np.isclose(Km[1, 0], 4.0 * (1 + np.sqrt(3)) * np.exp(-np.sqrt(3)))
print("all checks passed")

### Exercise 8.2b — Draw prior samples from each kernel

Before fitting, look at what each kernel *assumes*: draw three prior samples from each with plausible hyperparameters ($A = 0.7$; $\ell = 1.1$ d; $\Gamma = 1$, $P = 6$ d; $\ell_{\rm ev} = 15$ d), using the same random numbers in each panel.


In [ ]:
prior_kernels = {
    'SE':             lambda a, b: squared_exponential_kernel(a, b, 0.7, 1.1),
    'Matern 3/2':     lambda a, b: matern32_kernel(a, b, 0.7, 1.1),
    'Periodic':       lambda a, b: periodic_kernel(a, b, 0.7, 1.0, 6.0),
    'Quasi-periodic': lambda a, b: quasi_periodic_kernel(a, b, 0.7, 1.0, 6.0, 15.0),
}
z_common = np.random.default_rng(3).normal(size=(t_prior.size, 3))
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True)
for ax, (name, k) in zip(axes.ravel(), prior_kernels.items()):
    # TODO: L = cholesky(k(t_prior, t_prior) + 1e-8 I); plot L @ z_common
    ...
    ax.set_title(name); ax.grid(ls=':')
for ax in axes[-1]: ax.set_xlabel('time [days]')
plt.tight_layout()

## Exercise 8.3 — Fit the hyperparameters and predict

Maximise the log marginal likelihood with `scipy.optimize.minimize` (L-BFGS-B) over the **logarithms** of the hyperparameters, so they stay positive. The marginal likelihood is **multimodal in the period**, so for the periodic kernels start the optimiser from several initial periods between 3 and 10 d (by eye the cycles last ~6 d; a 'period' longer than the seasons would not be a periodicity at all) and keep the best. The dictionary below gives, for each kernel, a factory `theta -> kernel callable` and a list of starting points; you write the generic fitting loop.


In [ ]:
from scipy.optimize import minimize

kernel_specs = {
    'SE':             dict(make=lambda th: (lambda a, b: squared_exponential_kernel(a, b, *th)),
                           starts=[[0.7, 1.0]]),
    'Matern 3/2':     dict(make=lambda th: (lambda a, b: matern32_kernel(a, b, *th)),
                           starts=[[0.7, 1.0]]),
    'Periodic':       dict(make=lambda th: (lambda a, b: periodic_kernel(a, b, *th)),
                           starts=[[0.7, 1.0, P0] for P0 in [3.5, 4.5, 5.5, 6.5, 7.5, 9.0]]),
    'Quasi-periodic': dict(make=lambda th: (lambda a, b: quasi_periodic_kernel(a, b, *th)),
                           starts=[[0.7, 1.0, P0, 15.0] for P0 in [3.5, 4.5, 5.5, 6.5, 7.5, 9.0]]),
}
log_bounds = {'SE': [(-3, 2), (-2, 3)], 'Matern 3/2': [(-3, 2), (-2, 3)],
              'Periodic': [(-3, 2), (-2, 2), (np.log(3), np.log(10))],
              'Quasi-periodic': [(-3, 2), (-2, 2), (np.log(3), np.log(10)), (np.log(2), np.log(300))]}

def fit_kernel(name):
    """Return (best_theta, best_lml, kernel_callable) for kernel `name`, fitted on the training set."""
    spec = kernel_specs[name]
    def neg_lml(log_theta):
        try:
            return -log_marginal_likelihood_k(t_tr, y_tr, e_tr, spec['make'](np.exp(log_theta)))
        except np.linalg.LinAlgError:
            return 1e10
    best = None
    for th0 in spec['starts']:
        # TODO: res = minimize(neg_lml, np.log(th0), method='L-BFGS-B', bounds=log_bounds[name]); keep the best
        ...
    theta = np.exp(best.x)
    return theta, -best.fun, spec['make'](theta)

fits = {}
for name in kernel_specs:
    theta, lml, k = fit_kernel(name)
    mu, Cv = gp_predict_k(t_tr, y_tr, e_tr, t_prediction, k)
    fits[name] = dict(k=k, theta=theta, lml=lml, mu=mu, sd=np.sqrt(np.clip(np.diag(Cv), 0, None)))
    print(f"{name:15s} LML = {lml:7.2f}   theta = {np.round(theta, 3)}")


def predict_with(name, t):
    """Latent mean and std of fitted kernel `name` at times t."""
    mu, Cv = gp_predict_k(t_tr, y_tr, e_tr, np.atleast_1d(t), fits[name]['k'])
    return mu, np.sqrt(np.clip(np.diag(Cv), 0, None))

**Q8.3** Compare the fitted hyperparameters. Did the periodic kernels find a sensible period? Are the SE and Matérn length-scales short or long compared with that period, and what does that imply about how far those kernels can "see" across a gap?

*Your answer:*


## Exercise 8.4 — Posterior plots

Make a $2\times2$ figure: for each kernel show training points, withheld points, the posterior mean and the latent $\pm1\sigma$, $\pm2\sigma$ bands, with both gaps shaded.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
for ax, (name, f) in zip(axes.ravel(), fits.items()):
    # TODO: bands + mean + training points + test points (hint: plot_posterior draws ALL observations; draw test points on top in red)
    ...
    ax.axvspan(test_lo, test_hi, color='C3', alpha=0.1)
    ax.set_title(f"{name}   (train LML = {f['lml']:.1f})"); ax.set_ylim(-1.8, 1.8)
axes[0, 0].legend(loc='upper left', fontsize=9, ncol=2); plt.tight_layout()

**Q8.4** Describe what each kernel does (a) in the withheld interval, (b) in the natural gap, (c) beyond the ends of the data.

*Your answer:*


## Exercise 8.5 — Held-out scores

For the withheld points let $\mu_i$ be the predictive mean and $s_i^2 = \sigma_f^2(t_i) + \sigma_i^2$ the **total** predictive variance of a new measurement (Part 4!). Implement

1. **RMSE** $= \sqrt{\frac1N\sum (y_i-\mu_i)^2}$ — accuracy of the mean only;
2. **mean log predictive density** $= \frac1N\sum\log\mathcal N(y_i\mid\mu_i, s_i^2)$ — rewards accuracy *and* honest confidence;
3. **empirical coverage** — fraction with $|y_i-\mu_i| < s_i$ (nominal 0.68) and $< 1.96\,s_i$ (nominal 0.95);
4. the rms **standardised residual** $\sqrt{\frac1N\sum (y_i-\mu_i)^2/s_i^2}$ (should be ≈ 1) on the training points as well.


In [ ]:
from scipy.stats import norm

def heldout_scores(name, tt, yy, ee):
    mu, sd = predict_with(name, tt)
    s = np.sqrt(sd**2 + ee**2)
    # TODO
    rmse_  = ...
    lpd    = ...
    cov68  = ...
    cov95  = ...
    rms_z  = ...
    return rmse_, lpd, cov68, cov95, rms_z

print(f"{'kernel':15s} {'LML':>7s} | {'RMSE':>6s} {'LPD':>6s} {'cov68':>6s} {'cov95':>6s} {'rms z':>6s} | {'train rms z':>11s}")
for name in fits:
    r, l, c68, c95, z_te = heldout_scores(name, t_te, y_te, e_te)
    z_tr = heldout_scores(name, t_tr, y_tr, e_tr)[4]
    fits[name].update(rmse=r, lpd=l, c68=c68, c95=c95)
    print(f"{name:15s} {fits[name]['lml']:7.2f} | {r:6.3f} {l:6.2f} {c68:6.2f} {c95:6.2f} {z_te:6.2f} | {z_tr:11.2f}")
print(f"{'(nominal)':15s} {'':>7s} | {'':>6s} {'':>6s} {0.68:6.2f} {0.95:6.2f} {1.0:6.2f} | {1.0:11.2f}")

**Q8.5** Rank the kernels by training LML, by held-out RMSE and by held-out LPD. Do the rankings agree? With only ~8 withheld points the coverage fractions are coarse — which metric is most informative here, and why can a kernel be *calibrated but useless* or *useful but over-confident*?

*Your answer:*


## Exercise 8.6 — The missing season

The withheld interval is short. The real question is the 4.7-day natural gap, where there are no data but we know the truth. For each kernel compute the RMSE of the posterior mean against `true_signal` inside the gap and the fraction of the gap where the truth lies inside the $\pm2\sigma$ band, and plot the four posteriors zoomed on the gap with the truth overlaid.


In [ ]:
print(f"{'kernel':15s} {'RMSE vs truth (gap)':>20s} {'truth in 2sig':>14s} {'mean sigma_f':>13s}")
for name, f in fits.items():
    # TODO
    rmse_gap = ...
    inside   = ...
    print(f"{name:15s} {rmse_gap:20.3f} {inside:14.2f} {np.mean(f['sd'][gap_mask]):13.3f}")

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
for ax, (name, f) in zip(axes.ravel(), fits.items()):
    plot_posterior(f['mu'], f['sd'], ax=ax)
    ax.plot(t_prediction, true_signal, 'C3--', lw=1.3, label='hidden truth')
    ax.set_xlim(gap_start - 4, gap_end + 4); ax.set_ylim(-1.6, 1.6); ax.set_title(name)
axes[0, 0].legend(loc='upper left', fontsize=9); plt.tight_layout()

**Q8.6 (the key question)** A kernel may fit the observed points very well, but does it correctly predict the missing season? Answer for each kernel using the plot and the numbers. Which kernel would you trust to tell an observer what the star did during the gap, and how confident should that observer be?

*Your answer:*


**Q8.7** Reconsider Part 5. With the quasi-periodic kernel, where does the maximum-variance rule place the next observation, and is that still the most *useful* observation? What does this tell you about the interaction between kernel choice and observation planning?

*Your answer:*


**Q8.8** The training LML of the most flexible kernels is highest, yet we judged them on held-out data. Suppose you had a single dense night of 500 points and no gaps — would training fit alone be enough to choose a kernel? What does that say about choosing kernels by training likelihood?

*Your answer:*


---
# Take-home messages
- The kernel specifies statistical relationships between function values; it does not specify one unique curve.
- A covariance matrix is obtained by evaluating the kernel for every pair of input times; it is symmetric and positive semi-definite, and a small jitter keeps it numerically so.
- Prior functions share common statistical properties but differ as random realisations.
- Conditioning on observations gives a posterior mean and a posterior covariance, both computed with one Cholesky factorisation and triangular solves — never an explicit inverse.
- Measurement noise belongs in the training covariance; it is added again only when predicting a future *noisy* observation.
- Predictive variance depends on where and how well we observed, not on what we measured — which is what makes it usable for planning observations — but *which* places are uncertain depends on the kernel.
- Hyperparameters and the kernel itself are assumptions. Check them with the marginal likelihood and, above all, with held-out data that resemble the real prediction task.

### Source basis
This exercise is based on the Gaussian Process lecture materials supplied with the course, especially their treatment of covariance functions, prior samples, predictive equations, white noise, and active sampling.
